# Retrieval-Augmented Generation (RAG) & Vector Stores

This notebook covers the complete architecture of **Retrieval-Augmented Generation (RAG)** in LangChain, from vector embeddings and document chunking to vector databases and production LCEL retrieval chains.

---

## 1. The Context Architecture Matrix: System Prompt vs In-Context Learning vs RAG vs Fine-Tuning

| Dimension | System Prompting | In-Context Learning (Few-Shot) | Retrieval-Augmented Generation (RAG) | Fine-Tuning |
| :--- | :--- | :--- | :--- | :--- |
| **Primary Purpose** | Steer tone, persona, guardrails, role definition | Guide format and task execution with exemplars | Dynamically inject private, dynamic, and up-to-date knowledge | Adapt vocabulary, style, syntax, or niche domain nuances |
| **Knowledge Source** | Static instructions in context | In-context demonstration pairs | Dynamic external Vector DB / Documents / APIs | Parametric weights updated via gradient descent |
| **Knowledge Freshness** | Session-bound | Session-bound | Real-time (instant update upon indexing) | Static (requires re-training/checkpointing) |
| **Hallucination Risk** | High for factual domain specifics | Moderate | Low (grounded in retrieved context) | Moderate (can confidently hallucinate facts) |
| **Cost & Latency** | Low setup, standard inference tokens | Low setup, higher prompt token overhead | Moderate setup (indexing + embeddings + LLM prompt tokens) | High upfront training compute, lower per-prompt token cost |
| **Data Privacy** | Context payload sent to LLM provider | Sent in prompt | Filtered chunks sent via prompt | Training data baked into weights |

### Architectural Decision Framework:
* **Choose System Prompting** when defining model persona, safety guidelines, and conversational tone.
* **Choose In-Context Few-Shot** when demonstrating complex output formatting or specialized domain phrasing.
* **Choose RAG** when dealing with proprietary data, evolving knowledge bases, large enterprise documents, or when verifiable citations are strictly required.
* **Choose Fine-Tuning** when teaching the model a new syntax/language dialect, strict stylistic alignment, or reducing latency/tokens when the knowledge itself is static.

## 2. Environment Setup & Model Initialization

We initialize both the LLM generator (`ChatGoogleGenerativeAI`) and the dense embedding model (`GoogleGenerativeAIEmbeddings`). Fallbacks for OpenAI or HuggingFace embeddings are also supported.

In [ ]:
import os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

load_dotenv()

# Initialize Chat LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

# Initialize Embedding Model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004"
)

# Ensure docs directory is established
docs_dir = Path("docs")
docs_dir.mkdir(parents=True, exist_ok=True)

print("LLM Model:", llm.model)
print("Embedding Model:", embeddings.model)
print("Docs Directory:", docs_dir.resolve())

## 3. Understanding Vector Embeddings & Cosine Similarity

Vector embeddings project text into a high-dimensional dense vector space (typically 768 or 1536 dimensions) where semantic similarity corresponds to geometric proximity.

Cosine similarity between vectors $\mathbf{u}$ and $\mathbf{v}$ is given by:
$$\text{Cosine Similarity} = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

* Values close to `1.0` denote strong semantic alignment.
* Values near `0.0` denote orthogonal / unrelated content.

In [ ]:
def calculate_cosine_similarity(vec1, vec2):
    """Computes cosine similarity between two 1D numpy vectors."""
    a = np.array(vec1)
    b = np.array(vec2)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sentences = [
    "PostgreSQL is a powerful, open-source object-relational database system.",
    "Relational databases like Postgres store structured data in tables with ACID guarantees.",
    "The chef prepared a delicious wood-fired margherita pizza with fresh basil."
]

# Generate embedding vectors
vectors = embeddings.embed_documents(sentences)
print(f"Embedding Vector Dimensions: {len(vectors[0])} dimensions\n")

sim_0_1 = calculate_cosine_similarity(vectors[0], vectors[1])
sim_0_2 = calculate_cosine_similarity(vectors[0], vectors[2])

print(f"Similarity (Sentence 1 vs Sentence 2 - Semantically Related): {sim_0_1:.4f}")
print(f"Similarity (Sentence 1 vs Sentence 3 - Unrelated Topic):      {sim_0_2:.4f}")

## 4. Ingesting Files from `docs/` & Chunking with RecursiveCharacterTextSplitter

Chunking is crucial in RAG because:
1. **Granularity**: Smaller, coherent chunks provide precise context and minimize noise.
2. **Context Window Optimization**: Avoids stuffing irrelevant paragraphs into prompt tokens.
3. **Retrieval Accuracy**: Vector similarity degrades over excessively long documents.

`RecursiveCharacterTextSplitter` recursively splits by `["\n\n", "\n", " ", ""]` to preserve natural paragraph and sentence structure.

In [ ]:
from pathlib import Path

# Read source documents from docs/ directory
sec_file = Path("docs/sec_policy_204.txt")
ops_file = Path("docs/ops_policy_501.txt")

raw_corpus = [
    Document(
        page_content=sec_file.read_text(encoding="utf-8"),
        metadata={"source": "docs/sec_policy_204.txt", "category": "Security"}
    ),
    Document(
        page_content=ops_file.read_text(encoding="utf-8"),
        metadata={"source": "docs/ops_policy_501.txt", "category": "Operations"}
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

split_docs = splitter.split_documents(raw_corpus)
print(f"Original Documents: {len(raw_corpus)} -> Split Chunks: {len(split_docs)}")
for i, chunk in enumerate(split_docs):
    print(f"\n[Chunk {i+1} | Source: {chunk.metadata['source']} | Length: {len(chunk.page_content)} chars]")
    print(chunk.page_content)

## 5. Vector Store Indexing & Retriever Configuration

We index the split chunks into an `InMemoryVectorStore` (or `Chroma`) and convert it into a LangChain `Retriever` with `search_kwargs={"k": 2}`.

In [ ]:
# Index chunks into Vector Store
vectorstore = InMemoryVectorStore.from_documents(
    documents=split_docs,
    embedding=embeddings
)

# Convert into a Retriever to retrieve top 2 most relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Test retriever standalone
sample_query = "What is the procedure for emergency break glass access?"
retrieved_chunks = retriever.invoke(sample_query)

print(f"Query: '{sample_query}'")
print(f"Retrieved {len(retrieved_chunks)} relevant chunks:\n")
for doc in retrieved_chunks:
    print(f"Source: {doc.metadata['source']}")
    print(f"Content: {doc.page_content}\n---")

## 6. Complete LCEL RAG Chain with Source Grounding

We construct the complete retrieval pipeline:
1. `retriever | format_docs` fetches and formats the retrieved chunks into a clean context string.
2. `RunnablePassthrough()` passes the user's raw question.
3. The prompt formats both context and question into the system prompt.
4. The LLM generates the response, grounded strictly in the retrieved facts.
5. `StrOutputParser()` parses the completion string.

In [ ]:
# Helper function to format retrieved documents into context
def format_docs(docs):
    return "\n\n".join(
        f"[Document: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# Define RAG Prompt Template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an enterprise compliance AI assistant. 
Answer the user's question using ONLY the provided context below.
If the context does not contain the answer, state clearly: "I cannot answer based on the provided corporate documentation."
Always cite the source document name in your answer.

Context:
{context}"""),
    ("human", "{question}")
])

# Compose the LCEL RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test Grounded Query 1: Information present in docs
q1 = "When are engineering teams allowed to execute production deployments?"
print(f"QUESTION 1: {q1}\n")
print("ANSWER 1:")
print(rag_chain.invoke(q1))

# Test Out-of-Domain Query 2: Information NOT present in docs (Testing Hallucination Guardrail)
print("\n" + "="*60 + "\n")
q2 = "What is the annual compensation budget for cloud infrastructure engineers?"
print(f"QUESTION 2: {q2}\n")
print("ANSWER 2:")
print(rag_chain.invoke(q2))